# Week 20 Optional: Advanced Monitoring Patterns (SageMaker Studio Lab)

## Where this fits

You finished the main Week 20 notebook. The fraud system now has:

- SageMaker data capture on `fraud-classifier-endpoint`
- A default-baseline Data Quality Model Monitor schedule running hourly
- A pandas/scipy PSI drift check
- Langfuse + Strands OTEL traces on `week19_supervisor`
- A CloudWatch alarm on `ModelLatency`

This optional notebook extends four of those pieces. It is async. There are no in-class timings. Target audience: ML Engineer who wants production-grade depth.

## What you will build

1. RAGAS ONLINE evaluation - sample live retrieval traffic, score it with `Faithfulness` + `AnswerRelevancy` + `ContextPrecision` using Bedrock Haiku 3 as the judge, and push each score back into Langfuse so traces become a self-grading feedback loop
2. Langfuse datasets + custom dashboards - turn the Week 19/20 traces into a labeled eval dataset you can re-run on every model change, then walk through building a custom dashboard for cost, latency, and quality
3. SageMaker Model QUALITY monitor (advanced) - go beyond Data Quality. This one needs ground-truth labels and reports precision, recall, F1, and accuracy against the live `fraud-classifier-endpoint`
4. Evidently AI - same drift + classification quality story as Model Monitor, but open-source, portable, and runnable in any Python environment

## Prerequisites

- Main Week 20 notebook ran cleanly end-to-end.
- Endpoint `fraud-classifier-endpoint` is `InService`.
- KB `FARSQGTONR` is `ACTIVE` (you used it in Weeks 17-18 and the main Week 20 notebook).
- S3 bucket `bread-academy-week19-shared` is reachable.
- Langfuse keys are set in your Studio Lab environment (LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_HOST).


## Environment setup

You are on the same SageMaker Studio Lab kernel used for the main Week 20 notebook. The main notebook already pinned `sagemaker==2.257.3`, `langfuse>=2.50,<3`, `boto3>=1.35`, and the Strands stack. This notebook adds three new dependencies:

- `ragas>=0.2,<0.3` - RAGAS metrics for online evaluation
- `langchain-aws>=0.2` - `ChatBedrockConverse` and `BedrockEmbeddings` wrappers for the RAGAS judge
- `evidently>=0.4,<0.5` - the open-source drift / classification quality reporter
- `nest-asyncio>=1.5` - patches the Jupyter event loop so that RAGAS async scoring works inside notebook cells without RuntimeError

We deliberately keep `langfuse` on the v2 line. The Week 20 main notebook pinned `<3` for the same reason: v3 broke too many integrations and the score-attachment API is more stable on v2.

There is NO `dbutils`, NO `%pip` magic, NO Spark on SageMaker Studio Lab. We use `pip install` directly in a shell-escaped cell.


In [ ]:
# Install the optional deps. The main Week 20 notebook already installed
# sagemaker, langfuse, boto3, strands, etc. We only add what is new here.
# We also re-pin langfuse<3 defensively: the score(...) API used below is
# stable on v2; v3+ migrated to create_score(...).
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "ragas>=0.2,<0.3",
    "langchain-aws>=0.2",
    "evidently>=0.4,<0.5",
    "nest-asyncio>=1.5",
    "langfuse>=2.50,<3",
])

# Verify pinned versions. Use importlib.metadata, never __version__.
from importlib.metadata import version
print("sagemaker:    ", version("sagemaker"))
print("boto3:        ", version("boto3"))
print("langfuse:     ", version("langfuse"))
print("ragas:        ", version("ragas"))
print("langchain-aws:", version("langchain-aws"))
print("evidently:    ", version("evidently"))
print("nest-asyncio: ", version("nest-asyncio"))


In [ ]:
# SageMaker Studio Lab auth. No getpass, no dbutils.
import os
import boto3
import sagemaker
from sagemaker import get_execution_role

sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name
os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

# Langfuse keys are set as Studio environment variables by the instructor.
# RAGAS, LiteLLM, and the Langfuse SDK all read them directly from os.environ.
assert os.environ.get("LANGFUSE_PUBLIC_KEY"), "LANGFUSE_PUBLIC_KEY not set in Studio env"
assert os.environ.get("LANGFUSE_SECRET_KEY"), "LANGFUSE_SECRET_KEY not set in Studio env"
os.environ.setdefault("LANGFUSE_HOST", "https://cloud.langfuse.com")

# Carry-overs from main Week 20 / Week 17-18.
ENDPOINT_NAME = "fraud-classifier-endpoint"
BEDROCK_MODEL_ID = "us.anthropic.claude-3-haiku-20240307-v1:0"
EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"
KB_ID = "FARSQGTONR"
BUCKET = "bread-academy-week19-shared"

sagemaker_client = boto3.client("sagemaker", region_name=AWS_REGION)
bedrock_runtime  = boto3.client("bedrock-runtime", region_name=AWS_REGION)
bedrock_agent_rt = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)
cloudwatch       = boto3.client("cloudwatch", region_name=AWS_REGION)
s3               = boto3.client("s3", region_name=AWS_REGION)

print("Region:   ", AWS_REGION)
print("Role:     ", role)
print("Endpoint: ", ENDPOINT_NAME)
print("KB:       ", KB_ID)
print("Bucket:   ", BUCKET)


In [ ]:
# Fail loud if any upstream resource is missing.

# 1. Endpoint is InService.
resp = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
assert resp["EndpointStatus"] == "InService", (
    f"Endpoint {ENDPOINT_NAME} is {resp['EndpointStatus']}. "
    "Ask your instructor to redeploy the Week 19 endpoint."
)
print("Endpoint:", resp["EndpointStatus"])

# 2. Bedrock LLM probe.
try:
    bedrock_runtime.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "ping"}]}],
        inferenceConfig={"maxTokens": 10, "temperature": 0},
    )
    print("Bedrock LLM probe: ok")
except Exception:
    print("Ask your instructor to enable Bedrock access for", BEDROCK_MODEL_ID)
    raise

# 3. KB probe.
bedrock_agent = boto3.client("bedrock-agent", region_name=AWS_REGION)
kb = bedrock_agent.get_knowledge_base(knowledgeBaseId=KB_ID)
assert kb["knowledgeBase"]["status"] == "ACTIVE", f"KB {KB_ID} is {kb['knowledgeBase']['status']}"
print("KB:", kb["knowledgeBase"]["status"])

# 4. S3 bucket reachable.
s3.head_bucket(Bucket=BUCKET)
print("S3 bucket: ok")

# 5. Langfuse SDK can authenticate.
from langfuse import Langfuse
langfuse = Langfuse()  # reads env vars
print("Langfuse host:", os.environ["LANGFUSE_HOST"])


## Part 1 - RAGAS online evaluation

### Offline vs online RAG evaluation

In Week 18 you ran RAGAS on a fixed eval set: 20 hand-crafted questions, gold contexts, gold answers. That is OFFLINE evaluation. It catches regressions on the things you remembered to test. It does not catch:

- A new attack pattern the assistant has never seen.
- A subtle retrieval degradation after an embedding model change.
- Drift in the kinds of questions analysts actually ask in production.

ONLINE evaluation closes that gap. Sample (say) 5% of live traffic, run the same RAGAS metrics on it, and log each score back to Langfuse as a trace score. Now your dashboards have a real-time line for "faithfulness this hour" instead of "faithfulness on the dev set from three weeks ago".

### What we are scoring

Each fraud-assistant query produces:

- `user_input`: the analyst's question
- `retrieved_contexts`: the chunks the KB returned (from `bedrock_agent_rt.retrieve_and_generate`)
- `response`: the final LLM answer

We will compute three RAGAS metrics on each sampled query:

- `Faithfulness` - does the answer stay grounded in the retrieved contexts? (catches hallucinations)
- `AnswerRelevancy` - does the answer actually address the question? (catches off-topic drift)
- `ContextPrecision` - are the retrieved chunks relevant to the question? (catches retriever rot)

### The judge

RAGAS needs an LLM to grade these. We reuse the Week 18 pattern: `LangchainLLMWrapper(ChatBedrockConverse(model_id=BEDROCK_MODEL_ID))`. Haiku 3 is cheap enough that grading 5% of traffic is a rounding error on the inference bill.

### Cost guard

Online eval is async and sampled. NEVER run RAGAS on 100% of traffic in production. A 5% sample with three metrics adds roughly 3x cost on the sampled fraction, which works out to ~15% total LLM bill increase. Budget for it explicitly.


In [ ]:
# Build the RAGAS judge once. Reusable across all online scoring calls.

from langchain_aws import ChatBedrockConverse, BedrockEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
from ragas.dataset_schema import SingleTurnSample
import nest_asyncio
import asyncio
import uuid

# Jupyter kernels already run an event loop. nest_asyncio patches it so that
# asyncio.run() and .run_until_complete() work from inside a notebook cell.
nest_asyncio.apply()

judge_llm = LangchainLLMWrapper(
    ChatBedrockConverse(
        model_id=BEDROCK_MODEL_ID,
        region_name=AWS_REGION,
        temperature=0,
        max_tokens=512,
    )
)
judge_embed = LangchainEmbeddingsWrapper(
    BedrockEmbeddings(
        model_id=EMBED_MODEL_ID,
        region_name=AWS_REGION,
    )
)

faithfulness    = Faithfulness(llm=judge_llm)
answer_rel      = AnswerRelevancy(llm=judge_llm, embeddings=judge_embed)
context_prec    = ContextPrecision(llm=judge_llm)

# Issue ONE real KB-grounded query and score it end-to-end.
question = "What are the top three indicators of card-not-present fraud in our policy guide?"
trace_id = str(uuid.uuid4())  # we control the trace id so we can attach scores later

rag_resp = bedrock_agent_rt.retrieve_and_generate(
    input={"text": question},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": KB_ID,
            "modelArn": f"arn:aws:bedrock:{AWS_REGION}::foundation-model/{BEDROCK_MODEL_ID}",
        },
    },
)
answer = rag_resp["output"]["text"]
# Flatten retrievedReferences across all citations. The previous one-liner
# indexed citations[0] which crashes with IndexError when the response has
# zero citations (the model answered without citing any chunk). The nested
# comprehension below is safe for empty lists.
contexts = [c["content"]["text"]
            for cit in rag_resp.get("citations", [])
            for c in cit.get("retrievedReferences", [])]

sample = SingleTurnSample(
    user_input=question,
    response=answer,
    retrieved_contexts=contexts or ["(no retrieved context)"],
)

# RAGAS metrics are async. Run them serially here for clarity.
async def grade(sample):
    f  = await faithfulness.single_turn_ascore(sample)
    ar = await answer_rel.single_turn_ascore(sample)
    cp = await context_prec.single_turn_ascore(sample)
    return f, ar, cp

f, ar, cp = asyncio.get_event_loop().run_until_complete(grade(sample))
print(f"trace_id           : {trace_id}")
print(f"faithfulness       : {f:.3f}")
print(f"answer_relevancy   : {ar:.3f}")
print(f"context_precision  : {cp:.3f}")

# Push scores back to Langfuse so they show up on the trace.
langfuse.score(trace_id=trace_id, name="ragas_faithfulness",      value=float(f),  data_type="NUMERIC")
langfuse.score(trace_id=trace_id, name="ragas_answer_relevancy",  value=float(ar), data_type="NUMERIC")
langfuse.score(trace_id=trace_id, name="ragas_context_precision", value=float(cp), data_type="NUMERIC")
langfuse.flush()
print("Scores pushed to Langfuse. Open the trace to see them.")


### Lab 1 - Online RAGAS scoring on a 5-query sample

You will simulate one hour of fraud-assistant traffic (5 queries) and run online RAGAS scoring on EVERY query. In production you would sample 5% of traffic, not 100%; we score all 5 here so the lab finishes in a reasonable time.

Your task:

1. Use the `questions` list provided in the starter cell as your "live traffic".
2. For each question:
   a. Generate a fresh `trace_id` with `uuid.uuid4()`.
   b. Call `bedrock_agent_rt.retrieve_and_generate(...)` against `KB_ID`.
   c. Build a `SingleTurnSample` from the question / answer / contexts.
   d. Score it with all three metrics (`faithfulness`, `answer_rel`, `context_prec`).
   e. Push the three scores to Langfuse with `langfuse.score(trace_id=..., name=..., value=..., data_type="NUMERIC")`.
3. Collect the per-question scores into a list called `online_scores` where each entry is a dict with keys `question`, `trace_id`, `faithfulness`, `answer_relevancy`, `context_precision`.

Hints:
- Reuse the `grade(sample)` async helper from the demo.
- Call `langfuse.flush()` once at the end, not after every score.

### Homework extension

After Lab 1 runs, log in to Langfuse and:
1. Filter traces by score name `ragas_faithfulness` with value below 0.7. How many traces fall under that bar?
2. For each, read the answer text. Is the low score a real hallucination, or did the metric mis-grade a correct-but-terse answer? Write one sentence per trace.


In [ ]:
# Lab 1 starter. The line after each YOUR CODE marker must NOT reveal the answer.

questions = [
    "What is our threshold for flagging a transaction as high risk in real time?",
    "How does the policy handle disputed charges over $5000?",
    "What customer notification is required when a card is auto-frozen?",
    "Which merchant categories carry the highest fraud rate per our guide?",
    "What is the SLA for manual review queue clearance?",
]

online_scores = []

for q in questions:
    # YOUR CODE
    pass

print(f"Scored {len(online_scores)} queries.")
for entry in online_scores:
    print(entry)


In [ ]:
# SAFETY-NET for Lab 1. Run only if online_scores is empty so the rest of the
# notebook still works. SKIP this cell if you finished Lab 1.

if not online_scores:
    print("Using Lab 1 safety-net.")
    for q in questions:
        tid = str(uuid.uuid4())
        rag = bedrock_agent_rt.retrieve_and_generate(
            input={"text": q},
            retrieveAndGenerateConfiguration={
                "type": "KNOWLEDGE_BASE",
                "knowledgeBaseConfiguration": {
                    "knowledgeBaseId": KB_ID,
                    "modelArn": f"arn:aws:bedrock:{AWS_REGION}::foundation-model/{BEDROCK_MODEL_ID}",
                },
            },
        )
        ans = rag["output"]["text"]
        ctxs = [c["content"]["text"]
                for cit in rag.get("citations", [])
                for c in cit.get("retrievedReferences", [])]
        s = SingleTurnSample(user_input=q, response=ans, retrieved_contexts=ctxs or ["(no context)"])
        f, ar, cp = asyncio.get_event_loop().run_until_complete(grade(s))
        langfuse.score(trace_id=tid, name="ragas_faithfulness",      value=float(f),  data_type="NUMERIC")
        langfuse.score(trace_id=tid, name="ragas_answer_relevancy",  value=float(ar), data_type="NUMERIC")
        langfuse.score(trace_id=tid, name="ragas_context_precision", value=float(cp), data_type="NUMERIC")
        online_scores.append({
            "question": q, "trace_id": tid,
            "faithfulness": float(f), "answer_relevancy": float(ar), "context_precision": float(cp),
        })
    langfuse.flush()
    print(f"Scored {len(online_scores)} queries via safety-net.")


## Part 2 - Langfuse datasets and custom dashboards

### From traces to a re-usable eval set

Every trace Langfuse captures is potentially a regression test. The cleanest pattern is:

1. Engineer or analyst flags a trace as interesting (high score, low score, or a specific failure mode).
2. That trace becomes a `dataset_item` in a named Langfuse dataset.
3. Every model / prompt / KB change re-runs the full dataset and emits a `dataset_run` with the new outputs and scores.

This gives you exactly the same flow as a CI test suite for LLM applications. The Python SDK provides:

- `langfuse.create_dataset(name=...)` - idempotent.
- `langfuse.create_dataset_item(dataset_name=..., input=..., expected_output=..., metadata=...)` - the item.
- `langfuse.get_dataset(name=...)` - load items to iterate over.

We will turn the 5 sampled traces from Lab 1 into a `fraud-assistant-regression` dataset, with the `response` from Lab 1 as the "expected output" baseline.

### Custom dashboards

The default Langfuse dashboard shows traces over time and aggregate scores. Custom dashboards let you build operator-specific views:

- Cost per session by model
- p95 latency per route (which prompt template is slowest?)
- Faithfulness trend by KB version
- Error rate over time, split by trace tag

Dashboard widgets are configured in the UI under Dashboards -> New. Each widget queries one of: trace count, score values, observation latency, total cost, or a saved view. We will not build the dashboard programmatically (the API for dashboards is still evolving), but the notebook seeds enough data and tags so you can build one in 5 minutes in the UI. The Wrap-up section lists the exact widgets to add.


In [ ]:
# Idempotent dataset creation, then push the Lab 1 traces as items.

DATASET_NAME = "fraud-assistant-regression"

try:
    langfuse.create_dataset(
        name=DATASET_NAME,
        description="Regression eval set built from sampled live fraud-assistant traffic.",
        metadata={"source": "week20-optional", "kb_id": KB_ID},
    )
    print(f"Created dataset: {DATASET_NAME}")
except Exception as e:
    # create_dataset is idempotent server-side but the client may raise on duplicates
    # depending on SDK version. Either way, it now exists.
    print(f"Dataset already exists or duplicate-safe: {e!r}")

for entry in online_scores:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        input={"question": entry["question"]},
        expected_output=None,  # no human-graded gold yet; will be filled in by analyst review
        metadata={
            "source_trace_id": entry["trace_id"],
            "baseline_faithfulness": entry["faithfulness"],
            "baseline_answer_relevancy": entry["answer_relevancy"],
            "baseline_context_precision": entry["context_precision"],
        },
    )

langfuse.flush()
print(f"Pushed {len(online_scores)} items to dataset {DATASET_NAME!r}.")
print("Open Langfuse -> Datasets to review them and fill in `expected_output` for the top failures.")


### Lab 2 - Replay the dataset against the current KB and record a dataset run

A dataset is only useful if you can RE-RUN it. This lab replays every item against the live fraud assistant, scores each run, and tags every score with a `dataset_run` name so you can compare runs side-by-side in the UI.

Your task:

1. Load the dataset with `dataset = langfuse.get_dataset(DATASET_NAME)`.
2. For each `item` in `dataset.items`:
   a. Pull the `question` from `item.input`.
   b. Issue a fresh KB query (same `retrieve_and_generate` call as Lab 1).
   c. Score it with `faithfulness` only (one metric is enough for the lab; the homework runs all three).
   d. Link the score to the dataset run using `item.observe(run_name="run-2026-05-14-baseline")` as a context manager. Inside the `with` block, Langfuse will automatically associate the trace with this dataset run.
   e. Inside the `with` block, push the faithfulness score with `langfuse.score(trace_id=..., name="ragas_faithfulness", value=...)`.
3. After the loop, call `langfuse.flush()`.
4. Open Langfuse -> Datasets -> `fraud-assistant-regression` -> Runs. You should see one run named `run-2026-05-14-baseline` with 5 traces.

Hints:
- The `with item.observe(run_name=...) as trace:` pattern is the Langfuse SDK idiom for dataset runs in v2. `trace.id` gives you the trace_id to attach scores to.
- If `item.observe(...)` is unavailable in your SDK build, fall back to generating a uuid and passing `metadata={"langfuse_dataset_item_id": item.id, "langfuse_dataset_run_name": "run-2026-05-14-baseline"}` to the score call.

### Homework extension

After Lab 2 runs, change ONE thing in your KB query (e.g. drop `numberOfResults` from 5 to 2, or switch the `modelArn` to a different Bedrock model you have access to). Re-run the same dataset with `run_name="run-2026-05-14-fewer-chunks"`. Compare the two runs in the Langfuse Dataset Runs view. Which run has higher mean faithfulness? Was the difference statistically meaningful with only 5 items?


In [ ]:
# Lab 2 starter. The line after each YOUR CODE marker must NOT reveal the answer.

RUN_NAME = "run-2026-05-14-baseline"

dataset = None  # YOUR CODE

run_scores = []
for item in dataset.items if dataset else []:
    # YOUR CODE
    pass

print(f"Recorded {len(run_scores)} scored runs under {RUN_NAME!r}.")


In [ ]:
# SAFETY-NET for Lab 2. Run only if run_scores is empty.

if not run_scores:
    print("Using Lab 2 safety-net.")
    dataset = langfuse.get_dataset(DATASET_NAME)
    for item in dataset.items:
        q = item.input["question"]
        rag = bedrock_agent_rt.retrieve_and_generate(
            input={"text": q},
            retrieveAndGenerateConfiguration={
                "type": "KNOWLEDGE_BASE",
                "knowledgeBaseConfiguration": {
                    "knowledgeBaseId": KB_ID,
                    "modelArn": f"arn:aws:bedrock:{AWS_REGION}::foundation-model/{BEDROCK_MODEL_ID}",
                },
            },
        )
        ans = rag["output"]["text"]
        ctxs = [c["content"]["text"]
                for cit in rag.get("citations", [])
                for c in cit.get("retrievedReferences", [])]
        s = SingleTurnSample(user_input=q, response=ans, retrieved_contexts=ctxs or ["(no context)"])
        f = asyncio.get_event_loop().run_until_complete(faithfulness.single_turn_ascore(s))
        tid = str(uuid.uuid4())
        langfuse.score(
            trace_id=tid, name="ragas_faithfulness", value=float(f), data_type="NUMERIC",
            metadata={"dataset_item_id": item.id, "dataset_run_name": RUN_NAME},
        )
        run_scores.append({"question": q, "faithfulness": float(f), "trace_id": tid})
    langfuse.flush()
    print(f"Recorded {len(run_scores)} scored runs via safety-net.")


## Part 3 - SageMaker Model QUALITY monitor (advanced)

### Data Quality vs Model Quality

The main Week 20 notebook used `DefaultModelMonitor`, which is a DATA QUALITY monitor. It watches the distribution of input features and flags drift. It does NOT know if the predictions are correct.

`ModelQualityMonitor` is the next level up. It needs three things working together:

1. Endpoint data capture (you already have this).
2. A ground-truth manifest in S3 that maps each captured `inferenceId` back to its true label (fraud / not fraud) once the label is known. For a fraud system the ground truth typically arrives 7-30 days after the prediction, when chargebacks settle.
3. A `ModelQualityMonitor` schedule that joins (1) and (2) on `inferenceId`, computes precision/recall/F1/accuracy on the join, and emits CloudWatch metrics + a violations.json file.

### The ground-truth manifest format

SageMaker expects ground truth as JSON-lines in S3, one record per inference:

```json
{"groundTruthData": {"data": "1", "encoding": "CSV"}, "eventMetadata": {"eventId": "abc-123"}, "eventVersion": "0"}
```

Where `eventId` matches the `inferenceId` your application passed to `invoke_endpoint(... InferenceId="abc-123")`. The main Week 20 notebook did NOT pass `InferenceId`, so we will simulate ground truth here by generating IDs that match the inputs we send.

### The schedule

```python
from sagemaker.model_monitor import ModelQualityMonitor, EndpointInput
from sagemaker.model_monitor.dataset_format import DatasetFormat

mq = ModelQualityMonitor(role=role, instance_count=1, instance_type="ml.m5.xlarge",
                        sagemaker_session=sess)
mq.suggest_baseline(
    baseline_dataset=f"s3://{BUCKET}/week20-optional/baseline_quality.csv",
    dataset_format=DatasetFormat.csv(header=True),
    problem_type="BinaryClassification",
    inference_attribute="prediction",
    ground_truth_attribute="label",
)
mq.create_monitoring_schedule(
    monitor_schedule_name="fraud-classifier-model-quality-hourly",
    endpoint_input=EndpointInput(endpoint_name=ENDPOINT_NAME, ...),
    ground_truth_input=f"s3://{BUCKET}/week20-optional/ground-truth/",
    problem_type="BinaryClassification",
    ...
)
```

This is more setup than Data Quality, but the payoff is the metrics that actually matter to the business: how many real frauds did we catch this hour, and how many false positives did we generate.


In [ ]:
# Build a tiny baseline_quality.csv and a few ground-truth records in S3.
# This is the data plumbing a Model Quality schedule needs.

import csv
import io
import json

# 1. Baseline quality CSV - two columns: prediction (model output) and label (ground truth).
# These come from your hold-out set evaluated offline.
baseline_rows = [
    {"prediction": 1, "label": 1},
    {"prediction": 1, "label": 1},
    {"prediction": 0, "label": 0},
    {"prediction": 0, "label": 0},
    {"prediction": 1, "label": 0},  # false positive
    {"prediction": 0, "label": 1},  # false negative
    {"prediction": 1, "label": 1},
    {"prediction": 0, "label": 0},
]
buf = io.StringIO()
w = csv.DictWriter(buf, fieldnames=["prediction", "label"])
w.writeheader()
w.writerows(baseline_rows)
s3.put_object(
    Bucket=BUCKET,
    Key="week20-optional/baseline_quality.csv",
    Body=buf.getvalue().encode("utf-8"),
)
print("Baseline quality CSV uploaded.")

# 2. Ground-truth manifest in JSON-lines. eventId must match the InferenceId
# you would pass when calling invoke_endpoint(InferenceId=...).
gt_records = [
    {"groundTruthData": {"data": "1", "encoding": "CSV"},
     "eventMetadata": {"eventId": f"sim-event-{i}"},
     "eventVersion": "0"}
    for i in range(8)
]
gt_body = "\n".join(json.dumps(r) for r in gt_records).encode("utf-8")
s3.put_object(
    Bucket=BUCKET,
    Key="week20-optional/ground-truth/2026/05/14/ground-truth.jsonl",
    Body=gt_body,
)
print("Ground-truth manifest uploaded.")
print(f"S3 prefix: s3://{BUCKET}/week20-optional/ground-truth/")


### Lab 3 - Create a Model Quality monitoring schedule

You will create a `ModelQualityMonitor` schedule that reads from the live endpoint capture (already wired up in the main Week 20 notebook) and joins it against the ground-truth manifest you just uploaded.

Your task:

1. Construct a `ModelQualityMonitor(role=role, instance_count=1, instance_type="ml.m5.xlarge", sagemaker_session=sess)` instance.
2. Call `mq.suggest_baseline(...)` with:
   - `baseline_dataset=f"s3://{BUCKET}/week20-optional/baseline_quality.csv"`
   - `dataset_format=DatasetFormat.csv(header=True)`
   - `problem_type="BinaryClassification"`
   - `inference_attribute="prediction"`
   - `ground_truth_attribute="label"`
   - `output_s3_uri=f"s3://{BUCKET}/week20-optional/baseline-quality-results"`
3. Call `mq.create_monitoring_schedule(...)` with:
   - `monitor_schedule_name="fraud-classifier-model-quality-hourly"`
   - `endpoint_input=EndpointInput(endpoint_name=ENDPOINT_NAME, destination="/opt/ml/processing/input_data", inference_attribute="0", probability_attribute=None)`
   - `ground_truth_input=f"s3://{BUCKET}/week20-optional/ground-truth/"`
   - `problem_type="BinaryClassification"`
   - `output_s3_uri=f"s3://{BUCKET}/week20-optional/model-quality-results"`
   - `schedule_cron_expression=CronExpressionGenerator.hourly()`
   - `enable_cloudwatch_metrics=True`
4. Store the schedule name in a variable called `quality_schedule_name`.

Hint: import `ModelQualityMonitor`, `EndpointInput`, and `CronExpressionGenerator` from `sagemaker.model_monitor`. The role ARN is already in `role` from Cell 3.

### Homework extension

After Lab 3 runs (give it 60-90 minutes for the first scheduled job to fire), open the SageMaker console -> Model Monitor -> the schedule you created. Read the violations.json output. Which CloudWatch metrics did it emit? Cross-check that `precision`, `recall`, `f1`, and `accuracy` all appear in the `AWS/SageMaker/ModelQualityMonitoring` namespace and write a CloudWatch alarm on F1 dropping below 0.6.


In [ ]:
# Lab 3 starter. Build the ModelQualityMonitor, suggest a baseline, and create
# the schedule. Store the schedule name in `quality_schedule_name`.

from sagemaker.model_monitor import ModelQualityMonitor, EndpointInput, CronExpressionGenerator
from sagemaker.model_monitor.dataset_format import DatasetFormat

quality_schedule_name = None

# YOUR CODE

print("Quality schedule:", quality_schedule_name)


## Part 4 - Evidently AI as an open-source alternative

### Why an open-source alternative matters

SageMaker Model Monitor is fully managed. That is great when you want a hands-off solution, but it has tradeoffs:

- It is locked to AWS.
- It runs on a fixed schedule (hourly at minimum); ad-hoc analysis means clicking around the console.
- The reports are JSON-only; visualizations live in the SageMaker / CloudWatch UI.
- Debugging the schedule is painful (you wait for the next run, then read CloudWatch logs).

Evidently AI is pure Python, MIT-licensed, and runs anywhere. The same `Report(...).run(...)` call works on your laptop, in a Jupyter notebook, in a CI job, or as a SageMaker Processing Job triggered by EventBridge. It produces interactive HTML reports you can save and share.

### What we will build

A single Evidently `Report` with two presets:

- `DataDriftPreset()` - same idea as SageMaker Data Quality monitor: feature-by-feature distribution drift.
- `ClassificationPreset()` - same idea as SageMaker Model Quality monitor: precision, recall, F1, ROC curves, confusion matrix.

Input: two pandas DataFrames - a `reference_data` (training-time distribution + true labels) and a `current_data` (recent production sample + ground-truth labels we have so far). Both come from the same shared S3 bucket.

### Scheduling note

Evidently does NOT run on a schedule by itself. To run it periodically, the SageMaker-native equivalent of a Databricks Workflow is:

- EventBridge cron rule -> SageMaker Processing Job, OR
- SageMaker Pipelines with a Schedule.

We will not stand up the cron in this notebook (each student would need IAM to create EventBridge rules and that is noisy at 60-student scale). We document the pattern at the end so an ML Engineer can reproduce it in their own AWS account.


In [ ]:
# Generate a small reference + current pandas DataFrame from the shared bucket
# and produce one Evidently HTML report. This cell is BOTH the demo and the
# "lab" for Part 4 - it is fully worked because Evidently is straightforward
# once you have the DataFrames.

import pandas as pd
import numpy as np
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, ClassificationPreset

np.random.seed(42)

# Pretend reference (training-time) sample.
reference_data = pd.DataFrame({
    "transaction_amount":   np.random.gamma(2.0, 100.0, 1000),
    "hours_since_last_txn": np.random.gamma(2.0,  12.0, 1000),
    "prediction":           np.random.binomial(1, 0.10, 1000),
    "target":               np.random.binomial(1, 0.10, 1000),
})

# Pretend current (production) sample with a deliberate drift on transaction_amount
# and a degraded classifier.
current_data = pd.DataFrame({
    "transaction_amount":   np.random.gamma(2.0, 150.0, 1000),  # mean shifts up
    "hours_since_last_txn": np.random.gamma(2.0,  12.0, 1000),
    "prediction":           np.random.binomial(1, 0.18, 1000),  # over-predicts fraud
    "target":               np.random.binomial(1, 0.10, 1000),
})

report = Report(metrics=[
    DataDriftPreset(),
    ClassificationPreset(),
])
report.run(reference_data=reference_data, current_data=current_data)

# Save HTML so you can open it in Studio Lab via right-click -> Open in Browser.
out_path = "/tmp/evidently_fraud_report.html"
report.save_html(out_path)
print(f"Wrote: {out_path}")

# Also dump the JSON so you can pipe it into CloudWatch / Slack / a custom dashboard.
import json
report_json = report.as_dict()
print("Top-level metric keys in the report:")
for m in report_json.get("metrics", [])[:5]:
    print("-", m.get("metric"))

# Optional: upload the HTML to S3 so teammates can view it without Studio access.
s3.upload_file(out_path, BUCKET, "week20-optional/evidently/fraud_report.html")
print(f"Uploaded to: s3://{BUCKET}/week20-optional/evidently/fraud_report.html")


## Think About It

Take 5 minutes to reflect on these cross-topic questions. They have no single right answer.

1. You now have FOUR monitoring layers running on the same fraud system: SageMaker Data Quality (main Week 20), SageMaker Model Quality (Part 3), Langfuse + RAGAS online scores (Parts 1-2), and Evidently reports (Part 4). Two of those overlap heavily. If your team had to pick ONE to be the single source of truth for "is the fraud system healthy", which would you pick, and what would you do with the other three?

2. RAGAS online evaluation costs money on every sampled query. The Part 1 cost guard said "5% sampling, ~15% LLM bill increase". A VP at Bread Financial asks you to cut that cost in half while keeping the same coverage. List three concrete changes you would propose, ordered by which one you would try first.

3. The Part 3 Model Quality monitor needs ground-truth labels. For fraud, real labels arrive 7-30 days after the prediction. That means Model Quality is always reporting on month-old performance. What additional signals (from Parts 1, 2, or 4) can give you a real-time proxy for "the model is starting to fail", BEFORE ground truth confirms it?

## Wrap-up

In about 22 cells you closed four gaps from the main Week 20 notebook:

- RAGAS ONLINE evaluation grades live retrieval traffic and feeds the scores back to Langfuse, so you have real-time quality signals (not just stale offline eval).
- Langfuse datasets turn interesting traces into regression tests; custom dashboards make those scores operator-friendly.
- Model Quality monitor moves from "are inputs drifting?" to "are predictions still correct?" - which is what the business actually cares about.
- Evidently AI gives you the same drift + classification quality story in pure Python, runnable anywhere from your laptop to a Processing Job.

## Build the Langfuse custom dashboard (5-minute UI walkthrough)

After running this notebook, open Langfuse and:

1. Sidebar -> Dashboards -> New Dashboard. Name it `Fraud Assistant Health`.
2. Widget 1: Score over time. Filter score name = `ragas_faithfulness`. Aggregation = `mean`. Time bucket = `1 hour`.
3. Widget 2: Score distribution. Filter score name = `ragas_answer_relevancy`. Type = histogram.
4. Widget 3: Cost per trace. Filter tag contains `fraud-batch-`. Aggregation = `sum`.
5. Widget 4: p95 latency. Trace-level, group by trace name.
6. Save. Share the URL with your on-call team.

## Scheduling Evidently and RAGAS in production (no Databricks Workflows)

The SageMaker-native replacement for a Databricks scheduled notebook is one of:

- EventBridge cron rule -> `events.PutTargets` -> a Lambda that calls `sagemaker.create_processing_job(...)` with a container running this notebook's logic. Cheap, ~$0.05 per hourly run.
- SageMaker Pipelines with `PipelineSchedule` (newer, recommended for v2+). Same idea, slightly more managed UX.

In either case, the Processing Job container needs the same `pip install` block from Cell 2 and the same env vars from Cell 3.

## Further reading

- RAGAS metrics reference: https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/
- RAGAS + LangChain integration: https://docs.ragas.io/en/stable/howtos/integrations/langchain.html
- Langfuse datasets: https://langfuse.com/docs/evaluation/dataset-runs/datasets
- Langfuse custom dashboards: https://langfuse.com/docs/metrics/features/custom-dashboards
- SageMaker Model Quality monitor: https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-model-quality.html
- Ground-truth manifest format: https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-model-quality-merge.html
- Evidently AI quickstart: https://docs.evidentlyai.com/quickstart/classification_quality
- EventBridge + SageMaker Processing Jobs: https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job.html
